# Experiment 3: Timing Comparison with Equal Input and Hidden Widths

## George Bird, 2026

*Compares the inference and training times of four multilayer perceptron formulations across increasing network widths and depths, with input dimensions equal to hidden-layer widths:*

- **Standard MLP:** `NormalLinear` with `ElementwiseTanh`.
- **Ordinary isotropic MLP:** `NormalLinear` with `IsotropicTanh`.
- **Factorised isotropic MLP:** `FactorisedParallelLinear_FinalOptimised`, implementing the weight-chain factorisation of Eq. 12.
- **Reparameterised isotropic MLP:** `ReparameterisedParallelLinear_MatrixBiases`, implementing the generalised architecture of Sec. 3.1.

The ordinary and factorised isotropic architectures are mathematically equivalent formulations, whereas the reparameterised architecture introduces independently trainable weight and bias parameters, expanding the functional class.

**Architectures:** `[n, n, ..., n, 10]`, with widths `n ∈ {512, 1024, 2048, 4096}` and hidden-layer depths of 2, 4 and 6. Unlike Experiment 2, the input dimension varies with the hidden-layer width, allowing computational performance to be assessed when input and hidden dimensions are comparable.

**Data:** Randomly generated inputs of dimension `n` and integer class labels are used instead of real image datasets. Each benchmark uses 100 batches of 256 samples.

**Training:** Adam optimiser with learning rate `1e-3` and cross-entropy loss. Networks are compiled using `torch.compile(mode="reduce-overhead")`. Training times measure repeated calls to `training_epoch`, including forward passes, backpropagation and parameter updates.

**Inference:** Measures forward-pass execution without gradient computation. Factorised and reparameterised models use their specialised precomputed inference paths, allowing weight-input products to be evaluated in parallel. Compilation and warm-up operations are excluded from timing measurements.

**Repeats:** 20 timing repeats per architecture–formulation combination, with CUDA synchronisation before and after each timed measurement. Training repeats continue updating the same model rather than restarting optimisation.

**Evaluation:** Mean inference and training times, together with sample standard deviations, are reported across repeats to assess computational scaling with network width and depth. Particular attention is given to whether parallelised formulations outperform ordinary sequential MLPs as matrix multiplication becomes increasingly computationally dominant.

**Paper correspondence:** Section 4, Table 1 and Appendix A.2, Tables 2–3 of *Parallelising Matrix-Products in Isotropic Multilayer Perceptrons*.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import time
import io
from contextlib import redirect_stdout
import warnings
from Dependency import *
warnings.simplefilter("ignore")
torch.set_float32_matmul_precision("high")

In [ ]:
# Hyperparameters
LEARNING_RATE = 1e-3
BATCH_SIZE = 256
BATCHES_PER_EPOCH = 100
REPEATS = 20
DEVICE = try_gpu(output=True, i=0)
RANDOM_SEED = 1234
WIDTHS = [512, 1024, 2048, 4096]
DEPTHS = [2, 4, 6]

ARCHITECTURES = [
    [width] * (depth + 1) + [10]
    for width in WIDTHS
    for depth in DEPTHS
]

MODELS = {
    "Standard-tanh MLP": (NormalLinear, ElementwiseTanh),
    "Isotropic-tanh Ordinary MLP": (NormalLinear, IsotropicTanh),
    "Isotropic-tanh Weight-chain MLP": (FactorisedParallelLinear_FinalOptimised, IsotropicTanh),
    "Isotropic-tanh Reparameterised MLP": (ReparameterisedParallelLinear_MatrixBiases, IsotropicTanh),
}

torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

loss = nn.CrossEntropyLoss()

print(f"Device: {DEVICE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Batches per pass: {BATCHES_PER_EPOCH}")
print(f"Repeats: {REPEATS}")

In [ ]:
inference_results = {}
training_results = {}

for width_structure in ARCHITECTURES:
    architecture = str(width_structure)
    inference_results[architecture] = {}
    training_results[architecture] = {}

    random_batches = [
        (
            torch.randn(BATCH_SIZE, width_structure[0], device=DEVICE),
            torch.randint(0, 10, (BATCH_SIZE,), device=DEVICE),
        )
        for _ in range(BATCHES_PER_EPOCH)
    ]

    print(f"\n{architecture}")

    for model_name, (model_class, activation_class) in MODELS.items():
        torch.compiler.reset()

        network = model_class(width_structure, activation_class)
        network.reset_parameters()
        network.to(DEVICE)

        # Compile the normal forward path for training.
        compiled_network = torch.compile(
            network,
            mode="reduce-overhead"
        )

        # Compile the special prepared inference path separately.
        if hasattr(network, "prep_inference_time"):
            network.prep_inference_time()

            inference_function = torch.compile(
                network.forward_inference_time,
                mode="reduce-overhead"
            )
        else:
            inference_function = compiled_network

        network.eval()

        # Untimed inference compilation / warm-up.
        with torch.no_grad():
            for _ in range(3):
                inference_function(random_batches[0][0])

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        inference_times = []

        for repeat in range(REPEATS):
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()

            start_time = time.perf_counter()

            with torch.no_grad():
                for batch_data, _ in random_batches:
                    inference_function(batch_data)

            if DEVICE.type == "cuda":
                torch.cuda.synchronize()

            inference_times.append(
                time.perf_counter() - start_time
            )

            print(
                f"  {model_name} inference "
                f"{np.mean(inference_times):.4f} s "
                f"({repeat + 1}/{REPEATS})",
                end="\r"
            )

        inference_results[architecture][model_name] = inference_times

        # Reset before training.
        network.reset_parameters()

        optimiser = torch.optim.Adam(
            network.parameters(),
            lr=LEARNING_RATE
        )

        compiled_network.train()

        # Untimed training forward/backward compilation.
        batch_data, batch_labels = random_batches[0]

        optimiser.zero_grad(set_to_none=True)

        pred = compiled_network(batch_data)
        error = loss(pred, batch_labels)

        error.backward()
        optimiser.step()

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        # Remove the effect of the warm-up update.
        network.reset_parameters()
        network.zero_grad(set_to_none=True)

        optimiser = torch.optim.Adam(
            network.parameters(),
            lr=LEARNING_RATE
        )

        training_times = []

        for repeat in range(REPEATS):
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()

            start_time = time.perf_counter()

            with redirect_stdout(io.StringIO()):
                training_epoch(
                    compiled_network,
                    random_batches,
                    DEVICE,
                    optimiser,
                    loss,
                    repeat,
                    "classification",
                )

            if DEVICE.type == "cuda":
                torch.cuda.synchronize()

            training_times.append(
                time.perf_counter() - start_time
            )

            print(
                f"  {model_name} inference "
                f"{np.mean(inference_times):.4f} s "
                f"training {np.mean(training_times):.4f} s "
                f"({repeat + 1}/{REPEATS})",
                end="\r"
            )

        print()

        training_results[architecture][model_name] = training_times

In [ ]:
def mean_std(values):
    return f"{np.mean(values):.4f} ± {np.std(values, ddof=1):.4f}"


inference_table = pd.DataFrame({
    model_name: {
        architecture: mean_std(values[model_name])
        for architecture, values in inference_results.items()
    }
    for model_name in MODELS
})

training_table = pd.DataFrame({
    model_name: {
        architecture: mean_std(values[model_name])
        for architecture, values in training_results.items()
    }
    for model_name in MODELS
})

inference_table.index.name = "Architecture"
training_table.index.name = "Architecture"

print("Inference time (s)")
display(inference_table)

print("\nTraining time (s)")
display(training_table)

In [ ]:
print("\\begin{table}[H]")
print("\\centering")
print("\\begin{tabular}{ccccc}")
print("\\hline")
print("\\textbf{Architecture} & \\textbf{Standard-tanh} & \\textbf{Isotropic Ordinary} & \\textbf{Weight-chain} & \\textbf{Reparameterised} \\\\")
print("\\hline")

for architecture in inference_table.index:
    values = [inference_table.loc[architecture, model] for model in MODELS]
    print(f"{architecture} & " + " & ".join(values) + " \\\\")

print("\\hline")
print("\\end{tabular}")
print("\\caption{Inference time (s), mean $\\pm$ standard deviation across repeats.}")
print("\\end{table}")

print()

print("\\begin{table}[H]")
print("\\centering")
print("\\begin{tabular}{ccccc}")
print("\\hline")
print("\\textbf{Architecture} & \\textbf{Standard-tanh} & \\textbf{Isotropic Ordinary} & \\textbf{Weight-chain} & \\textbf{Reparameterised} \\\\")
print("\\hline")

for architecture in training_table.index:
    values = [training_table.loc[architecture, model] for model in MODELS]
    print(f"{architecture} & " + " & ".join(values) + " \\\\")

print("\\hline")
print("\\end{tabular}")
print("\\caption{Training epoch time (s), mean $\\pm$ standard deviation across repeats.}")
print("\\end{table}")